In [2]:
# !wget https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip -O data_original.zip
# !unzip data_original.zip

--2023-11-21 21:00:49--  https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/irsafilo/KION_DATASET/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip [following]
--2023-11-21 21:00:49--  https://raw.githubusercontent.com/irsafilo/KION_DATASET/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 78795295 (75M) [application/zip]
Saving to: ‘data_original.zip’

data_original.zip   100%[===================>]  75.14M   234MB/s    in 0.3s    

In [3]:
# !pip install rectools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.0/99.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 57.9 MB/s eta 0:00:00


In [11]:
import time

from typing import Dict, Any, List
from collections.abc import KeysView
from copy import deepcopy
import numpy as np
import pandas as pd
from tqdm import tqdm
from pprint import pprint
import matplotlib.pyplot as plt
import seaborn as sns

from implicit.nearest_neighbours import TFIDFRecommender

from rectools import Columns

from rectools.dataset import Interactions, Dataset
from rectools.metrics import Precision, Recall, MeanInvUserFreq, Serendipity, calc_metrics
from rectools.metrics.ranking import MAP, MRR, NDCG
from rectools.models import RandomModel, PopularModel
from rectools.models.base import ModelBase
from rectools.metrics.base import MetricAtK
from rectools.model_selection import TimeRangeSplitter
from rectools.model_selection.splitter import Splitter

### Расчет метрик

In [12]:

def calculate_metrics(
    models: Dict[str, ModelBase],
    metrics: Dict[str,MetricAtK],
    splitter: Splitter,
    K: int,
    data: pd.DataFrame
) -> pd.DataFrame:
    """
    Функция для расчёта метрик с использованием кросс-валидации.

    Аргументы:
    - models: Словарь с инициализированными моделями.
    - metrics: Словарь с инициализированными метриками.
    - splitter: Объект Splitter для кросс-валидации.
    - K: Количество рекомендаций для генерации.
    - data: Исходный датасет.

    Возвращает:
    - DataFrame: Усреднённые результаты метрик по моделям и метрикам.
    """
    interactions = Interactions(data)
    splitter.get_test_fold_borders(interactions)
    dataset = Dataset.construct(data)
    results = []
    fold_iterator = splitter.split(interactions, collect_fold_stats=True)

    for train_ids, test_ids, fold_info in tqdm((fold_iterator), total=splitter.n_splits):
        print(f"\n==================== Fold {fold_info['i_split']}")
        pprint(fold_info)

        df_train = interactions.df.iloc[train_ids]
        dataset = Dataset.construct(df_train)

        df_test = interactions.df.iloc[test_ids][Columns.UserItem]
        test_users = np.unique(df_test[Columns.User])

        # Catalog is set of items that we recommend.
        # Sometimes we recommend not all items from train.
        catalog = df_train[Columns.Item].unique()

        for model_name, model in models.items():
            model.fit(dataset)
            recos = model.recommend(
                users=test_users,
                dataset=dataset,
                k=K,
                filter_viewed=True,
            )
            metric_values = calc_metrics(
                metrics,
                reco=recos,
                interactions=df_test,
                prev_interactions=df_train,
                catalog=catalog,
            )
            res = {"fold": fold_info["i_split"], "model": model_name}
            res.update(metric_values)
            results.append(res)

    pivot_results = pd.DataFrame(results).drop(columns="fold").groupby(["model"], sort=False).agg(["mean", "std"])
    mean_metric_subset = [(metric, agg) for metric, agg in pivot_results.columns if agg == 'mean']
    pivot_results = (
        pivot_results.style
        .highlight_min(subset=mean_metric_subset, color='lightcoral', axis=0)
        .highlight_max(subset=mean_metric_subset, color='lightgreen', axis=0)
    )
    return pivot_results

In [21]:
def visualize_recommendations(
    model: ModelBase,
    data: pd.DataFrame,
    user_ids: List[int],
    item_data: pd.DataFrame,
    K: int
    ) -> None:
    reco = model.recommend(
    users=user_ids,
    dataset=Dataset.construct(data),
    k=K,
    filter_viewed=True,
    )
    for user in user_ids:
        print(f"Visualization for User ID: {user}")

        # История просмотров пользователя
        user_history = (
            data[data.user_id == user]
            .merge(item_data, on='item_id')
            .sort_values(by='datetime', ascending=False)
        )

        # Рекомендации для пользователя
        user_recommendations = (
            reco[reco.user_id == user]
            .merge(item_data, on='item_id')
        )

        # Вывод истории просмотров
        print("\nUser History:")
        display(user_history)

        # Вывод рекомендаций
        print("\nUser Recommendations:")
        display(user_recommendations)

### Тестирование

In [14]:
models = {
    'RandomModel': RandomModel(random_state=32),
    'PopularModel': PopularModel()
}

In [15]:
metrics = {
    "Precision@1": Precision(k=1),
    "Precision@5": Precision(k=5),
    "Precision@10": Precision(k=10),

    "Recall@1": Recall(k=1),
    "Recall@5": Recall(k=5),
    "Recall@10": Recall(k=10),

    "MAP@1": MAP(k=1),
    "MAP@5": MAP(k=5),
    "MAP@10": MAP(k=10),

    "MRR@1": MRR(k=1),
    "MRR@5": MRR(k=5),
    "MRR@10": MRR(k=10),

    "NDCG@1": NDCG(k=1),
    "NDCG@5": NDCG(k=5),
    "NDCG@10": NDCG(k=10),

    "Novelty@1": MeanInvUserFreq(k=1),
    "Novelty@5": MeanInvUserFreq(k=5),
    "Novelty@10": MeanInvUserFreq(k=10),

    "Serendipity@1": Serendipity(k=1),
    "Serendipity@5": Serendipity(k=5),
    "Serendipity@10": Serendipity(k=10),
}

In [16]:
splitter = TimeRangeSplitter(
    test_size="7D",
    n_splits=3,
    filter_already_seen=True,
    filter_cold_items=True,
    filter_cold_users=True,
)

In [17]:
path_data = "data_original"

interactions_df = pd.read_csv(
    path_data + "/" + "interactions.csv",
    sep=",",
    names=[Columns.User, Columns.Item, Columns.Datetime, Columns.Weight, Columns.Score],
)

interactions_df = interactions_df.iloc[1:]
interactions_df = interactions_df[[Columns.User, Columns.Item, Columns.Datetime, Columns.Score]]
interactions_df[Columns.Weight] = 1.0

interactions_df["item_id"] = interactions_df["item_id"].astype("int")
interactions_df[Columns.User] = interactions_df[Columns.User].astype("int")

print(interactions_df.shape)
interactions_df.head()

<ipython-input-17-6b7c50389f03>:3: DtypeWarning: Columns (0,1,3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  interactions_df = pd.read_csv(


(5476251, 5)


,user_id,item_id,datetime,score,weight
1,176549,9506,2021-05-11,72.0,1.0
2,699317,1659,2021-05-29,100.0,1.0
3,656683,7107,2021-05-09,0.0,1.0
4,864613,7638,2021-07-05,100.0,1.0
5,964868,9506,2021-04-30,100.0,1.0


In [18]:
calculate_metrics(models, metrics, splitter, K=10, data=interactions_df)

  0%|          | 0/3 [00:00<?, ?it/s]


==================== Fold 0
{'end': Timestamp('2021-08-09 00:00:00', freq='7D'),
 'i_split': 0,
 'start': Timestamp('2021-08-02 00:00:00', freq='7D'),
 'test': 263681,
 'test_items': 6602,
 'test_users': 98184,
 'train': 4266013,
 'train_items': 15237,
 'train_users': 797423}


 33%|███▎      | 1/3 [00:35<01:10, 35.47s/it]


==================== Fold 1
{'end': Timestamp('2021-08-16 00:00:00', freq='7D'),
 'i_split': 1,
 'start': Timestamp('2021-08-09 00:00:00', freq='7D'),
 'test': 279422,
 'test_items': 6698,
 'test_users': 103511,
 'train': 4649162,
 'train_items': 15415,
 'train_users': 850489}


 67%|██████▋   | 2/3 [01:13<00:37, 37.01s/it]


==================== Fold 2
{'end': Timestamp('2021-08-23 00:00:00', freq='7D'),
 'i_split': 2,
 'start': Timestamp('2021-08-16 00:00:00', freq='7D'),
 'test': 298878,
 'test_items': 6679,
 'test_users': 110076,
 'train': 5051815,
 'train_items': 15577,
 'train_users': 906071}


100%|██████████| 3/3 [01:56<00:00, 38.86s/it]


In [19]:
%%time
items_df = pd.read_csv(
    path_data + "/" + "items.csv",
    sep=",",
)
items_df.head()

CPU times: user 664 ms, sys: 30.8 ms, total: 695 ms
Wall time: 697 ms


,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [22]:
user_ids = [666262, 672861, 955527]

# Проведение визуального анализа
for model_name, model in models.items():
    print(f"Visualization for {model_name}")
    res =  visualize_recommendations(model, interactions_df, user_ids, items_df, K=10)
    print(res)

Visualization for RandomModel
Visualization for User ID: 666262

User History:


,user_id,item_id,datetime,score,weight,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
2,666262,12981,2021-05-14,100.0,1.0,film,Томирис,Tomiris,2020.0,"боевики, драмы, историческое, военные",Казахстан,NaN,16.0,NaN,Акан Сатаев,"Альмира Турсын, Адиль Ахметов, Берик Айтжанов,...","Среди всех древних народов, населяющих террито...","2020, казахстан, томирис"
0,666262,7957,2021-05-12,32.0,1.0,film,Последний викинг,The Lost Viking,2018.0,"боевики, историческое, приключения",Великобритания,NaN,16.0,NaN,Эммет Кумминс,"Дин Ридж, Росс О’Хеннесси, Кезия Берроуз, Джей...",852 год. Викинги завоевывают и грабят земли по...,"викинг, 2018, соединенное королевство, последний"
1,666262,4785,2021-05-12,28.0,1.0,film,Робин Гуд: Начало,Robin Hood,2018.0,"боевики, триллеры, приключения",США,NaN,16.0,NaN,Отто Батхёрст,"Джейми Фокс, Ф. Мюррэй Абрахам, Тэрон Эджертон...",Ветеран Крестового похода лорд Робин Локсли во...,"шериф, Робин Гуд, лучник, вор, стимпанк, Нотти..."



User Recommendations:


,user_id,item_id,score,rank,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,666262,10101,10,1,series,Возвращение Будулая,Vozvrashchenie Budulaya,1985.0,мелодрамы,Россия,NaN,12.0,NaN,Александр Бланк,"Михай Волонтир, Клара Лучко, Ариадна Шенгелая,...",Более полугода не возвращалась память к жесток...,NaN
1,666262,619,9,2,film,Новые приключения Аладдина (жестовым языком),The New Adventures of Aladdin (Sign Language),2016.0,"зарубежные, комедии","Бельгия, Франция",NaN,6.0,NaN,Артур Бензакен,"Мишель Блан, Жан-Поль Рув, Эрик Жюдор, Рамзи Б...",Новая версия бессмертной арабской сказки о при...,"Новые, приключения, Аладдина, жестовым, языком..."
2,666262,12618,8,3,film,Пропавшая грамота,Propavshaya gramota,1972.0,"фэнтези, комедии",СССР,NaN,12.0,NaN,Борис Ивченко,"Иван Миколайчук, Василий Хорошко, Галина Долго...","Героическая народная комедия, где с лукавой ус...","Украина, казак, казаки, флаг, Ведьма, Казак, 1..."
3,666262,5967,7,4,series,Братья вне игры,Spitsbroers (The Score),2015.0,"драмы, спорт",Россия,0.0,18.0,NaN,"Гийс Польспоель, Йорун Дюмулейн","Оскар Уиллемс, Йорен Селдеслахтс, Луис Тэйлп, ...",Два брата мечтают об одном — попасть в мир бол...,"Братья, вне, игры, 2015, Россия"
4,666262,4041,6,5,film,Фрилансеры,Freelancers,2012.0,"криминал, детективы, драмы, зарубежные, боевики",США,NaN,18.0,NaN,Джесси Терреро,"Анабель Акоста, Бо Гарретт, Майкл МакГрэйди, М...",Криминальный боевик о сыне убитого нью-йоркско...,"Фрилансеры, 2012, США, бандиты, гангстеры, кор..."
5,666262,5701,5,6,film,Алые паруса: Новая история,NaN,2019.0,"комедии, мелодрамы",Казахстан,NaN,12.0,NaN,NaN,"Айнур Ниязова, Бибигуль Актан, Василий Уриевск...",Мечтательная девушка из глубинки Асель отправл...,"Алые, паруса, Новая, история, 2019, Казахстан,..."
6,666262,9738,4,7,series,Женщина в беде 3,A Woman in Trouble 3,2016.0,"детективы, мелодрамы",Россия,NaN,12.0,NaN,Алексей Гусев,"Максим Щеголев, Татьяна Казючиц, Кира Кауфман",Мирная семейная жизнь Ани и Ивана нарушена быв...,"Женщина, беде, 3, 2016, Россия"
7,666262,15247,3,8,film,Гордость и предубеждение,Pride and Prejudice,1940.0,"драмы, мелодрамы",США,NaN,12.0,NaN,Роберт З. Леонард,"Грир Гарсон, Лоуренс Оливье, Мэри Боланд, Эдна...","Мистер и миссис Беннет, состоятельная аристокр...","1940, соединенные штаты, гордость, предубеждение"
8,666262,10004,2,9,film,Болванчики,Bobbleheads: The Movie,2020.0,"мультфильм, приключения, комедии",США,NaN,6.0,NaN,Кирк Уайз,"Шер, Дженнифер Кулидж, Энтони Дестефанис, Хала...",Этот забавный и трогательный фильм обязательно...,"2020, соединенные штаты, болванчики"
9,666262,2816,1,10,film,Избави нас от лукавого,Deliver Us from Evil,2014.0,"ужасы, триллеры, детективы",США,NaN,18.0,NaN,Скотт Дерриксон,"Эрик Бана, Эдгар Рамирес, Оливия Манн, Крис Ко...",Полиция Нью-Йорка расследует серию тревожных и...,"детектив, исповедь, пещера, лев, зоопарк, библ..."


Visualization for User ID: 672861

User History:


...

Visualization for User ID: 955527

User History:


,user_id,item_id,datetime,score,weight,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,955527,1183,2021-06-02,1.0,1.0,film,Стань легендой! Бигфут Младший,The Son of Bigfoot,2017.0,"мультфильм, фэнтези, приключения, комедии",Франция,NaN,6.0,NaN,"Жереми Дегрусон, Бэн Стассен","Синда Адамс, Джордж Бэббит, Лайла Берзиньш, Дж...",Обычный подросток по имени Адам отправляется н...,"йети, злодей, лес, отец, лаборатория, антропом..."
2,955527,4725,2021-06-02,4.0,1.0,film,Лобановский навсегда,Lobanovskiy Forever,2016.0,"спорт, биография, документальное",Украина,NaN,16.0,NaN,Антон Азаров,"Карло Анчелотти, Олег Блохин, Мишель Платини, ...",Фильм расскажет о легендарном тренере киевског...,"украина, футбол, лобановский, Гений, Оммаж, Тр..."
3,955527,1238,2021-06-02,7.0,1.0,film,Диего Марадона,Diego Maradona,2019.0,"спорт, биография, документальное",Великобритания,NaN,16.0,NaN,Азиф Кападиа,"Диего Армандо Марадона, Диего Марадона, Диего ...",Харизматичный аргентинец Диего Марадона был ге...,"архивные кадры, футболист, футбол, спортивный ..."
1,955527,13371,2021-05-04,11.0,1.0,film,Пеле: Рождение легенды,Pele: Birth of a Legend(aka Pele),2016.0,"драмы, спорт, биография",США,NaN,12.0,NaN,"Джефф Цимбалист, Майкл Цимбалист","Кевин де Паула, Леонардо Лима Карвальо, Сеу Жо...",Биографическая драма расскажет о закулисье спо...,"биография, спорт, профессиональный футболист, ..."



User Recommendations:


,user_id,item_id,score,rank,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,955527,4398,189923.0,1,series,Муж на час,NaN,2014.0,"мелодрамы, комедии",Украина,NaN,12.0,NaN,Анатолий Матешко,"Ярослав Бойко, Мария Куликова, Юрий Горбунов, ...","Главный герой работал в НИИ, конструировал нов...",NaN
1,955527,11754,180487.0,2,film,Kingsman: Секретная служба,Kingsman: The Secret Service,2015.0,"боевики, криминал, приключения, комедии","Великобритания, США",NaN,18.0,NaN,Мэттью Вон,"Тэрон Эджертон, Колин Фёрт, Сэмюэл Л. Джексон,...","Эггси — молодой парень, который прошел службу ...","шпион, великобритания, секретная организация, ..."
2,955527,3813,119797.0,3,film,Опасное погружение,Pressure,2015.0,"драмы, триллеры, приключения",Великобритания,NaN,16.0,NaN,Рон Скальпелло,"Дэнни Хьюстон, Мэттью Гуд, Джо Коул, Алан МакК...",Маленькая капсула с четырьмя водолазами застря...,"выживание, подводное плавание, 2015, соединенн..."
3,955527,16228,115095.0,4,series,Содержанки,NaN,2021.0,триллеры,Россия,0.0,18.0,NaN,"Константин Богомолов, Дарья Жук, Юрий Мороз","Дарья Мороз, Софья Эрнст, Сергей Бурунов, Влад...","Тонкое исследование того, как и чем живёт стол...","Содержанки, 2021, Россия"
4,955527,6563,85914.0,5,series,Бриллианты для Джульетты,Brillianty dlya Dzhulyetty,2004.0,комедии,Россия,NaN,12.0,NaN,Валерий Чиков,"Татьяна Арнтгольц, Алиса Гребенщикова, Антон К...","Три студента — Паша, Кеша и Лева — почти не пь...",NaN
5,955527,13980,69687.0,6,film,Изгой-один: Звёздные войны. Истории.,Rogue One: A Star Wars Story,2016.0,"боевики, фантастика, приключения",США,NaN,16.0,NaN,Гарет Эвардс,"Фелисити Джонс, Диего Луна, Алан Тьюдик, Донни...",Сопротивление собирает отряд для выполнения ос...,"бунтарь, космический корабль, космическое сраж..."
6,955527,12215,66415.0,7,film,Недетское кино,Not Another Teen Movie,2001.0,"мелодрамы, комедии",США,NaN,18.0,NaN,Джоэл Галлен,"Кайлер Ли, Крис Эванс, Джейми Прессли, Эрик Кр...",Популярный в школе парень и звезда футбольной ...,"аутсайдер, мяч, поцелуй, старшая школа, школьн..."
7,955527,11505,53191.0,8,series,Бездомный Бог,Noragami,2014.0,"аниме, фэнтези, приключения, комедии",Япония,NaN,12.0,NaN,"Тамура Котаро, Цуёси Хида, Сюдзи Мияхара","Хироси Камия, Маая Утида, Юки Кадзи, Аки Тоёса...",Малоизвестный Бог без собственного храма решае...,"аниме, боги, божество, духи умерших, оригиналь..."
8,955527,8491,42877.0,9,film,Адский бункер,Outpost,2007.0,"боевики, ужасы, фантастика",Великобритания,NaN,18.0,NaN,Стив Баркер,"Рэй Стивенсон, Джулиан Уэдэм, Ричард Брэйк, По...",Международный отряд наемников сопровождает уче...,"бункер, нацист, восточная европа, наемник, отк..."
9,955527,3281,39498.0,10,series,Вольф Мессинг: Видевший сквозь время,Messing,2009.0,"драмы, историческое",Россия,NaN,12.0,NaN,"Владимир Краснопольский, Валерий Усков","Евгений Князев, Тара Амирханова, Михаил Горево...","Мессинг родился в последний год 19 века, чтобы...",NaN


None
